# **1. Perkenalan Dataset**

Notebook ini merupakan adaptasi dari `Template_Eksperimen_MSML.ipynb` untuk proyek **Sistem Klasifikasi Performa Belajar Siswa** berbasis Open University Learning Analytics Dataset (OULAD).

Dataset OULAD berisi informasi demografi siswa, registrasi kelas, assessment, course, dan interaksi siswa pada Virtual Learning Environment (VLE). Pada proyek ini, dataset digunakan untuk membangun sistem klasifikasi risiko/performa belajar siswa sebagai dasar adaptive learning dan early warning system.

**Sumber dataset:**

Kuzilek, J., Hlosta, M., & Zdrahal, Z. (2015). Open University Learning Analytics dataset. UCI Machine Learning Repository. https://doi.org/10.24432/C5KK69

**Target klasifikasi:**

Kolom `final_result` dari `studentInfo.csv` dipetakan menjadi target binary:

- `Pass` dan `Distinction` menjadi `Good`
- `Fail` dan `Withdrawn` menjadi `At_Risk`

Dengan pemetaan ini, model memprediksi apakah siswa berada pada kondisi performa baik atau berisiko membutuhkan intervensi pembelajaran.


Tahap pertama adalah memastikan dataset memenuhi kebutuhan submission dan relevan dengan masalah machine learning yang diselesaikan. Dataset OULAD dipilih karena memiliki data aktivitas LMS/VLE yang sesuai dengan konteks sistem pembelajaran digital.


# **2. Import Library**

Pada tahap ini, pustaka yang dibutuhkan untuk eksplorasi dataset dan preprocessing diimpor. Notebook ini hanya menggunakan pustaka umum seperti `pandas`, `numpy`, `matplotlib`, dan `seaborn`. Script preprocessing otomatis berada pada `automate_yesaya.py`.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in [start, *start.parents]:
        raw_dir = candidate / "dataset" / "open+university+learning+analytics+dataset"
        experiment_dir = candidate / "Eksperimen_SML_Yesaya"
        if raw_dir.exists() and experiment_dir.exists():
            return candidate
    raise FileNotFoundError("Project root tidak ditemukan. Jalankan notebook dari dalam folder project.")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "dataset" / "open+university+learning+analytics+dataset"
EXPERIMENT_DIR = PROJECT_ROOT / "Eksperimen_SML_Yesaya"
PREPROCESSING_DIR = EXPERIMENT_DIR / "preprocessing"
OUTPUT_DIR = EXPERIMENT_DIR / "open_UL_analysis_preprocessing"
OUTPUT_FILE = OUTPUT_DIR / "student_cleaned.csv"
METADATA_FILE = OUTPUT_DIR / "preprocessing_metadata.json"

PROJECT_ROOT, RAW_DIR, OUTPUT_FILE


# **3. Memuat Dataset**

Pada tahap ini, tabel-tabel utama OULAD dimuat. File `studentVle.csv` berukuran besar, sehingga pada tahap eksplorasi notebook hanya membaca sampel awal. Proses agregasi penuh terhadap `studentVle.csv` dilakukan oleh script otomatis `automate_yesaya.py` menggunakan chunk processing.


In [ ]:
csv_files = {
    "courses": RAW_DIR / "courses.csv",
    "assessments": RAW_DIR / "assessments.csv",
    "student_info": RAW_DIR / "studentInfo.csv",
    "student_registration": RAW_DIR / "studentRegistration.csv",
    "student_assessment": RAW_DIR / "studentAssessment.csv",
    "vle": RAW_DIR / "vle.csv",
    "student_vle": RAW_DIR / "studentVle.csv",
}

for name, path in csv_files.items():
    if not path.exists():
        raise FileNotFoundError(f"File dataset tidak ditemukan: {path}")

courses = pd.read_csv(csv_files["courses"], na_values=["?"])
assessments = pd.read_csv(csv_files["assessments"], na_values=["?"])
student_info = pd.read_csv(csv_files["student_info"], na_values=["?"])
student_registration = pd.read_csv(csv_files["student_registration"], na_values=["?"])
student_assessment = pd.read_csv(csv_files["student_assessment"], na_values=["?"])
vle = pd.read_csv(csv_files["vle"], na_values=["?"])
student_vle_sample = pd.read_csv(csv_files["student_vle"], nrows=1000, na_values=["?"])

file_summary = pd.DataFrame(
    [
        {
            "table": name,
            "file_name": path.name,
            "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
        }
        for name, path in csv_files.items()
    ]
)

shape_summary = pd.DataFrame(
    [
        {"table": "courses", "rows_loaded": len(courses), "columns": courses.shape[1]},
        {"table": "assessments", "rows_loaded": len(assessments), "columns": assessments.shape[1]},
        {"table": "student_info", "rows_loaded": len(student_info), "columns": student_info.shape[1]},
        {"table": "student_registration", "rows_loaded": len(student_registration), "columns": student_registration.shape[1]},
        {"table": "student_assessment", "rows_loaded": len(student_assessment), "columns": student_assessment.shape[1]},
        {"table": "vle", "rows_loaded": len(vle), "columns": vle.shape[1]},
        {"table": "student_vle_sample", "rows_loaded": len(student_vle_sample), "columns": student_vle_sample.shape[1]},
    ]
)

print("Ringkasan ukuran file:")
display(file_summary)
print("Ringkasan data yang dimuat:")
display(shape_summary)
student_info.head()


# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini dilakukan EDA untuk memahami distribusi target, komposisi modul, missing value, dan karakter umum data. EDA ini menjadi dasar keputusan preprocessing dan feature engineering.


In [ ]:
target_map = {
    "Pass": "Good",
    "Distinction": "Good",
    "Fail": "At_Risk",
    "Withdrawn": "At_Risk",
}
student_info = student_info.copy()
student_info["target_label"] = student_info["final_result"].map(target_map)

print("Distribusi final_result asli:")
display(student_info["final_result"].value_counts().rename("count").to_frame())

print("Distribusi target binary:")
display(student_info["target_label"].value_counts().rename("count").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=student_info, x="final_result", ax=axes[0], order=student_info["final_result"].value_counts().index)
axes[0].set_title("Distribusi final_result Asli")
axes[0].set_xlabel("final_result")
axes[0].set_ylabel("Jumlah")

sns.countplot(data=student_info, x="target_label", ax=axes[1], order=["At_Risk", "Good"])
axes[1].set_title("Distribusi Target Binary")
axes[1].set_xlabel("target_label")
axes[1].set_ylabel("Jumlah")
plt.tight_layout()
plt.show()

missing_summary = (
    student_info.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_percent=lambda df: (df["missing_count"] / len(student_info) * 100).round(2))
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)
print("Missing value pada studentInfo.csv:")
display(missing_summary)

module_summary = (
    student_info.groupby(["code_module", "code_presentation", "target_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print("Distribusi target per module-presentation:")
display(module_summary.head(15))

numeric_columns = ["num_of_prev_attempts", "studied_credits"]
print("Statistik fitur numerik awal:")
display(student_info[numeric_columns].describe())

print("Contoh data aktivitas VLE:")
display(student_vle_sample.head())


# **5. Data Preprocessing**

Tahap preprocessing dilakukan secara otomatis melalui `automate_yesaya.py`. Script tersebut menjalankan langkah berikut:

1. Membaca raw dataset OULAD.
2. Membuat target binary `Good` dan `At_Risk`.
3. Menggabungkan data siswa, course, registrasi, assessment, dan aktivitas VLE.
4. Mengagregasi assessment menjadi fitur seperti jumlah submission, skor rata-rata, weighted score, dan jumlah late submission.
5. Mengagregasi aktivitas VLE menjadi fitur seperti total klik, active days, klik sebelum course mulai, klik 30 hari pertama, dan klik 60 hari pertama.
6. Menangani missing value dengan kategori `Unknown` untuk kategorikal dan median untuk numerik.
7. Melakukan one-hot encoding fitur kategorikal.
8. Menyimpan dataset bersih ke `open_UL_analysis_preprocessing/student_cleaned.csv`.

Bagian ini menjalankan script otomatis dan memvalidasi output akhir.


In [ ]:
if str(PREPROCESSING_DIR) not in sys.path:
    sys.path.insert(0, str(PREPROCESSING_DIR))

from automate_yesaya import main as run_preprocessing

run_preprocessing()

cleaned = pd.read_csv(OUTPUT_FILE)
metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))

print("Metadata preprocessing:")
display({
    "rows": metadata["rows"],
    "columns": metadata["columns"],
    "feature_count": metadata["feature_count"],
    "target_column": metadata["target_column"],
    "target_distribution": metadata["target_distribution"],
})

print("Validasi dataset bersih:")
print(f"Shape: {cleaned.shape}")
print(f"Total missing value: {int(cleaned.isna().sum().sum())}")
print(f"Kolom target tersedia: {'target' in cleaned.columns}")
print(f"Kolom target_label tersedia: {'target_label' in cleaned.columns}")

display(cleaned["target_label"].value_counts().rename("count").to_frame())
display(cleaned.head())
